# ATIS → JSON conversion

Turn the raw ATIS CSVs into the curated structured-output JSON records used for fine-tuning.

**Schema** (fixed, 8 keys; `null` when the query doesn't mention the field):
```json
{"intent": ["flight"], "from_city": null, "to_city": null,
 "depart_date": null, "depart_time": null, "airline": null,
 "round_trip": null, "class_type": null}
```

**Intent rule:** keep a query only if *every* component of its intent is in
`{flight, airfare, ground_service, airline}`. Intent is a list normalized to a canonical order.

In [1]:
import json
import pandas as pd

DATA_DIR = "data"

# Chosen intents (canonical order used to normalize compound intents).
INTENT_ORDER = ["flight", "airfare", "ground_service", "airline"]
CHOSEN = set(INTENT_ORDER)

# Map an ATIS slot *family* (text before the first dot) to a schema field.
FIELD_MAP = {
    "fromloc": "from_city",
    "toloc": "to_city",
    "depart_date": "depart_date",
    "depart_time": "depart_time",
    "airline_name": "airline",
    "airline_code": "airline",
    "round_trip": "round_trip",
    "class_type": "class_type",
}
SCHEMA_FIELDS = ["from_city", "to_city", "depart_date", "depart_time",
                 "airline", "round_trip", "class_type"]

In [2]:
def normalize_intent(intent_str):
    """Return a canonical-ordered list of intent components, or None if any
    component is outside the chosen set (query should be dropped)."""
    parts = set(intent_str.split("+"))
    if not parts <= CHOSEN:
        return None
    return [i for i in INTENT_ORDER if i in parts]


def extract_spans(text, slots):
    """Group BIO tags into (base_slot, span_text) pairs, in order of appearance."""
    tokens, tags = text.split(), slots.split()
    spans, cur_base, cur_toks = [], None, []
    for tok, tag in zip(tokens, tags):
        if tag == "O":
            if cur_base is not None:
                spans.append((cur_base, " ".join(cur_toks)))
                cur_base, cur_toks = None, []
            continue
        prefix, base = tag[0], tag[2:]
        if prefix == "B" or base != cur_base:  # new span (lenient on stray I-)
            if cur_base is not None:
                spans.append((cur_base, " ".join(cur_toks)))
            cur_base, cur_toks = base, [tok]
        else:  # continuation
            cur_toks.append(tok)
    if cur_base is not None:
        spans.append((cur_base, " ".join(cur_toks)))
    return spans


def to_record(text, slots):
    """Build the curated field dict (without intent) for one query.
    All spans of a field are joined in order, so multi-token values like
    "june 12" or "after 6 pm" are kept whole."""
    acc = {f: [] for f in SCHEMA_FIELDS}
    for base, span_text in extract_spans(text, slots):
        field = FIELD_MAP.get(base.split(".")[0])
        if field is not None:
            acc[field].append(span_text)
    return {f: " ".join(v) if v else None for f, v in acc.items()}

In [3]:
def build(csv_path):
    df = pd.read_csv(csv_path)
    rows, dropped = [], 0
    for _, r in df.iterrows():
        intent = normalize_intent(r["intent"])
        if intent is None:
            dropped += 1
            continue
        record = {"intent": intent, **to_record(r["text"], r["slots"])}
        rows.append({"query": r["text"], "output": record})
    return rows, dropped


train_rows, train_dropped = build(f"{DATA_DIR}/atis_train.csv")
test_rows, test_dropped = build(f"{DATA_DIR}/atis_test.csv")
print(f"train: kept {len(train_rows)}, dropped {train_dropped}")
print(f"test:  kept {len(test_rows)}, dropped {test_dropped}")
print(f"total kept: {len(train_rows) + len(test_rows)}")

train: kept 4522, dropped 456
test:  kept 768, dropped 125
total kept: 5290


## Inspect a few records

In [4]:
for row in train_rows[:5]:
    print("Q:", row["query"])
    print(json.dumps(row["output"], indent=2))
    print("-" * 60)

Q: i want to fly from boston at 838 am and arrive in denver at 1110 in the morning
{
  "intent": [
    "flight"
  ],
  "from_city": "boston",
  "to_city": "denver",
  "depart_date": null,
  "depart_time": "838 am",
  "airline": null,
  "round_trip": null,
  "class_type": null
}
------------------------------------------------------------
Q: what flights are available from pittsburgh to baltimore on thursday morning
{
  "intent": [
    "flight"
  ],
  "from_city": "pittsburgh",
  "to_city": "baltimore",
  "depart_date": "thursday",
  "depart_time": "morning",
  "airline": null,
  "round_trip": null,
  "class_type": null
}
------------------------------------------------------------
Q: cheapest airfare from tacoma to orlando
{
  "intent": [
    "airfare"
  ],
  "from_city": "tacoma",
  "to_city": "orlando",
  "depart_date": null,
  "depart_time": null,
  "airline": null,
  "round_trip": null,
  "class_type": null
}
------------------------------------------------------------
Q: round tri

## Field fill-rates and intent distribution
Confirms how sparse the output is (how often each field is non-null) and the kept intent mix.

In [5]:
all_rows = train_rows + test_rows
outputs = pd.DataFrame([r["output"] for r in all_rows])

fill = (outputs[SCHEMA_FIELDS].notna().mean() * 100).round(1).sort_values(ascending=False)
print("field fill-rate (% non-null):")
print(fill.to_string())

print("\nintent distribution (as joined label):")
intent_label = outputs["intent"].apply(lambda x: "+".join(x))
print(intent_label.value_counts().to_string())

field fill-rate (% non-null):
from_city      90.5
to_city        89.5
depart_date    28.0
depart_time    18.8
airline        14.9
round_trip      7.7
class_type      3.8

intent distribution (as joined label):
intent
flight            4298
airfare            471
ground_service     291
airline            195
flight+airfare      34
flight+airline       1


## Save as JSONL
Written under `data/` (gitignored). Each line: `{"query": ..., "output": {...}}`.

In [6]:
def save_jsonl(rows, path):
    with open(path, "w") as f:
        for row in rows:
            f.write(json.dumps(row) + "\n")
    print(f"wrote {len(rows)} -> {path}")

save_jsonl(train_rows, f"{DATA_DIR}/atis_train.jsonl")
save_jsonl(test_rows, f"{DATA_DIR}/atis_test.jsonl")

wrote 4522 -> data/atis_train.jsonl
wrote 768 -> data/atis_test.jsonl
